# Shared US trading calendar
Prices are loaded through `00_engine.load()` before percentage changes. Non-US sessions are excluded before computing returns, preserving valid returns immediately after US holidays. Fee rates and evaluation periods also come from the shared engine.


**Submission reference:** Quote historical final-portfolio performance and comparisons from the `Submission reference: historical IS / OOS metrics` table in `results/Report.html` (`results/tables/final_metrics.csv`, generated by 07). Identify the strategy row and period. This notebook's standalone performance tables are research diagnostics; their figures must not be mixed with the final account metrics.


In [ ]:
# ============================================================
# V2 - Step 1: InitializeBSVDynamic defensive allocation experiment
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project root
ROOT = Path.cwd().resolve()
while not (ROOT / "pipeline" / "00_engine.ipynb").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent   # From pipeline can locate the project root from any subdirectory

# One shared calendar, fee table, and period definitions for both risk stages.
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from portfolio_accounting import ENGINE
IS_START = ENGINE['IS_START']


print("Project root:", ROOT)

# ETFTrading universe
ETFS = ["TQQQ", "UVXY", "TECL", "SPXL", "TECS", "BSV"]

# Main parameters
LOOKBACK = 60
ALPHA = 1.0
TARGET_VOL = 0.30
TRADING_DAYS = 252

# andV1Consistent backtest start date
IS_START = ENGINE['IS_START']

# LoadV1Exported daily weights
weights_path = (
    ROOT / "results/signals/risk_adjusted_weights.csv"
)

v1_weights = pd.read_csv(
    weights_path,
    index_col=0,
    parse_dates=True
)[ETFS]

# Load original ensemble weights (Subsequently withV2Compare)
raw_path = (
    ROOT / "results/signals/ensemble_weights_history.csv"
)

raw_weights = pd.read_csv(
    raw_path,
    index_col=0,
    parse_dates=True
)[ETFS]

# LoadETFPrice
price_path = ROOT / "data/raw/etf_daily.csv"

prices = ENGINE['load'](price_path)  # Filter US sessions BEFORE returns.

# Use adjusted closing prices to compute returns
adj_close = prices["Adj Close"][ETFS]

daily_returns = adj_close.pct_change(fill_method=None)

# Retain sixETFTrading days with all returns available
valid_returns = daily_returns[ETFS].dropna()

# Data checks
assert v1_weights.index.equals(raw_weights.index)
assert np.allclose(v1_weights.sum(axis=1), 1.0)
assert v1_weights.notna().all().all()

print("\n========== Data checks ==========")
print("Original ensemble weights: ", raw_weights.shape)
print("V1Risk-adjusted weights: ", v1_weights.shape)
print("ETFHistorical returns: ", valid_returns.shape)

print("\n========== Backtest dates ==========")
print("Start date: ", v1_weights.index.min())
print("End date: ", v1_weights.index.max())

print("\n========== V1Average weights ==========")
display(
    (v1_weights.mean() * 100)
    .round(2)
    .to_frame("Average Weight (%)")
)

print("\n========== V1Latest weights ==========")
display(
    (v1_weights.tail(1).T * 100)
    .round(2)
)

In [ ]:
# ============================================================
# V2 - Step 2: Calculate daily estimated portfolio volatility
# ============================================================

portfolio_vol = pd.Series(
    index=v1_weights.index,
    dtype=float,
    name="Forecast Portfolio Volatility"
)

for date in v1_weights.index:

    # Through the current date, Historical60trading days of returns
    past_returns = (
        valid_returns
        .loc[:date, ETFS]
        .tail(LOOKBACK)
    )

    assert len(past_returns) == LOOKBACK, (
        f"{date}: Insufficient historical data"
    )

    # 60day return covariance matrix
    cov_matrix = past_returns.cov().values

    # Same dayV1Weights
    w = v1_weights.loc[date, ETFS].values

    # Estimated portfolio annualized volatility
    portfolio_vol.loc[date] = np.sqrt(
        TRADING_DAYS * (w @ cov_matrix @ w)
    )


# ------------------------------------------------------------
# Check results
# ------------------------------------------------------------

assert portfolio_vol.notna().all()
assert (portfolio_vol > 0).all()

print("========== V1Estimated portfolio volatility ==========")

display(
    portfolio_vol.describe()
    .to_frame("Annualized Volatility")
    .round(4)
)

print("\n========== Most recent5trading days ==========")

display(
    (portfolio_vol.tail(5) * 100)
    .round(2)
    .to_frame("Forecast Vol (%)")
)

print("\n========== Exceeds30%Proportion of trading days above target ==========")

above_target = (portfolio_vol > TARGET_VOL).mean()

print(f"{above_target:.2%}")

print("\n========== Latest forecast volatility ==========")

print(f"{portfolio_vol.iloc[-1]:.2%}")

In [ ]:
# ============================================================
# V2 - Step 3: DynamicBSVDefensive allocation
# ============================================================

# 1. Calculate risk exposure scaling factors
risk_scale = (
    TARGET_VOL / portfolio_vol
).clip(upper=1.0)

# Check scaling factors
assert risk_scale.notna().all()
assert ((risk_scale > 0) & (risk_scale <= 1)).all()

# 2. CopyV1Weights
v2_weights = v1_weights.copy()

# 3. Scale only non-BSVAssets
RISK_ASSETS = [etf for etf in ETFS if etf != "BSV"]

v2_weights[RISK_ASSETS] = (
    v1_weights[RISK_ASSETS]
    .mul(risk_scale, axis=0)
)

# 4. Allocate all released capital toBSV
v2_weights["BSV"] = (
    1.0 - v2_weights[RISK_ASSETS].sum(axis=1)
)

# ------------------------------------------------------------
# 5. Data checks
# ------------------------------------------------------------

assert v2_weights.notna().all().all()

assert (v2_weights >= -1e-10).all().all()

assert np.allclose(
    v2_weights.sum(axis=1),
    1.0
), "Some total weights differ from100%dates"

assert (
    v2_weights["BSV"] >= v1_weights["BSV"] - 1e-10
).all(), "ObservedBSVUnexpected reduction in weight"

print("========== V2Average weights ==========")

display(
    pd.DataFrame({
        "V1 (%)": v1_weights.mean() * 100,
        "V2 (%)": v2_weights.mean() * 100
    }).round(2)
)

# ------------------------------------------------------------
# 6. Latest weight comparison
# ------------------------------------------------------------

print("\n========== Latest trading day weights ==========")

latest_comparison = pd.DataFrame({
    "V1 (%)": v1_weights.iloc[-1] * 100,
    "V2 (%)": v2_weights.iloc[-1] * 100
})

display(latest_comparison.round(2))

# ------------------------------------------------------------
# 7. Defensive mechanism triggers
# ------------------------------------------------------------

print("\n========== BSVDefensive mechanism statistics ==========")

print(
    f"Proportion of trading days triggering risk scaling: "
    f"{(risk_scale < 1).mean():.2%}"
)

print(
    f"Average risk exposure multiplier: "
    f"{risk_scale.mean():.2%}"
)

print(
    f"V1AverageBSVWeights: "
    f"{v1_weights['BSV'].mean():.2%}"
)

print(
    f"V2AverageBSVWeights: "
    f"{v2_weights['BSV'].mean():.2%}"
)

print(
    f"V2HighestBSVWeights: "
    f"{v2_weights['BSV'].max():.2%}"
)

In [ ]:
# ============================================================
# V2 - Step 4: Compare addingBSVEstimated portfolio volatility before and after
# ============================================================

# 1. CalculateV2Daily estimated volatility
v2_portfolio_vol = pd.Series(
    index=v2_weights.index,
    dtype=float,
    name="V2 Forecast Volatility"
)

for date in v2_weights.index:

    # Historical observations through that days close60trading day returns
    past_returns = (
        valid_returns
        .loc[:date, ETFS]
        .tail(LOOKBACK)
    )

    # Calculate covariance matrix
    cov_matrix = past_returns.cov().values

    # RetrieveV2Weights
    w = v2_weights.loc[date, ETFS].values

    # Calculate forecast annualized portfolio volatility
    v2_portfolio_vol.loc[date] = np.sqrt(
        TRADING_DAYS * (w @ cov_matrix @ w)
    )


# ------------------------------------------------------------
# 2. Data checks
# ------------------------------------------------------------

assert v2_portfolio_vol.notna().all()
assert (v2_portfolio_vol >= 0).all()


# ------------------------------------------------------------
# 3. V1 vs V2Volatility comparison
# ------------------------------------------------------------

vol_comparison = pd.DataFrame({
    "V1": portfolio_vol,
    "V2": v2_portfolio_vol
})

print("========== V1 vs V2Estimated volatility ==========")

display(
    (vol_comparison.describe() * 100)
    .round(2)
)


# ------------------------------------------------------------
# 4. Average volatility
# ------------------------------------------------------------

print("\n========== Average forecast annualized volatility ==========")

print(f"V1: {portfolio_vol.mean():.2%}")
print(f"V2: {v2_portfolio_vol.mean():.2%}")

print(
    f"Reduction: "
    f"{(portfolio_vol.mean() - v2_portfolio_vol.mean()):.2%}"
)


# ------------------------------------------------------------
# 5. Exceeds30%Proportion above target
# ------------------------------------------------------------

print("\n========== Exceeds30%Proportion of trading days above target ==========")

print(
    f"V1: {(portfolio_vol > TARGET_VOL).mean():.2%}"
)

print(
    f"V2: {(v2_portfolio_vol > TARGET_VOL).mean():.2%}"
)


# ------------------------------------------------------------
# 6. Most recent5trading days
# ------------------------------------------------------------

print("\n========== Most recent5day estimated volatility ==========")

display(
    (vol_comparison.tail(5) * 100)
    .round(2)
)


# ------------------------------------------------------------
# 7. Latest trading day
# ------------------------------------------------------------

print("\n========== Latest trading day ==========")

print(f"V1: {portfolio_vol.iloc[-1]:.2%}")
print(f"V2: {v2_portfolio_vol.iloc[-1]:.2%}")


# ------------------------------------------------------------
# 8. Forecast volatility is lower every day?
# ------------------------------------------------------------

lower_vol_days = (
    v2_portfolio_vol < portfolio_vol - 1e-10
).mean()

print("\n========== Proportion of trading days with lower risk ==========")

print(f"{lower_vol_days:.2%}")

In [ ]:
# ============================================================
# V2 - Step 5: V0 / V1 / V2 Backtest performance comparison
# ============================================================

import json
import ast

# ------------------------------------------------------------
# 1. Load the existing team backtest function
# ------------------------------------------------------------

# Shared exact transaction-cost solver used by the extracted portfolio function.
_engine_ns = ENGINE
rebalance_exact = ENGINE['rebalance_exact']

def load_function(notebook_path, function_name):

    with open(notebook_path, encoding="utf-8") as f:
        notebook = json.load(f)

    for cell in notebook["cells"]:

        if cell["cell_type"] != "code":
            continue

        source = "".join(cell["source"])

        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue

        for node in tree.body:

            if (
                isinstance(node, ast.FunctionDef)
                and node.name == function_name
            ):

                module = ast.Module(
                    body=[node],
                    type_ignores=[]
                )

                exec(
                    compile(module, "<notebook>", "exec"),
                    globals()
                )

                return

    raise ValueError(f"Cannot find function: {function_name}")


load_function(
    next((ROOT / "pipeline").rglob("02_parameter_ensemble.ipynb")),
    "backtest_weighted_portfolio"
)

load_function(
    ROOT / "pipeline/00_engine.ipynb",
    "metrics"
)

print("Backtest and performance functions loaded successfully")


# ------------------------------------------------------------
# 2. Use the original team transaction costs
# ------------------------------------------------------------

TICKS = ETFS

ONE_WAY_BPS = {t: ENGINE['ONE_WAY_BPS'][t] for t in ETFS}


# ------------------------------------------------------------
# 3. Construct adjusted opening and closing prices on the same basis as before
# ------------------------------------------------------------

price_data = prices.sort_index()

adjustment_factor = (
    price_data["Adj Close"][ETFS]
    / price_data["Close"][ETFS]
)

open_adjusted = (
    price_data["Open"][ETFS]
    * adjustment_factor
)

O = open_adjusted.reindex(raw_weights.index)[ETFS]

C = (
    price_data["Adj Close"][ETFS]
    .reindex(raw_weights.index)
)

assert O.notna().all().all()
assert C.notna().all().all()


# ------------------------------------------------------------
# 4. Backtest three strategies
# ------------------------------------------------------------

strategies = {
    "V0 Original": raw_weights,
    "V1 Inverse Vol": v1_weights,
    "V2 BSV Targeting": v2_weights
}

equity_results = {}
diagnostics = {}

for name, weights in strategies.items():

    equity, diag = backtest_weighted_portfolio(
        O,
        C,
        weights,
        start=IS_START
    )

    equity_results[name] = equity
    diagnostics[name] = diag

    print(
        f"{name}: "
        f"Final Equity = {equity.iloc[-1]:.4f}"
    )


# ------------------------------------------------------------
# 5. Check reproducibility 05 notebooks V0 / V1 Backtest
#    Previously compared against hard-coded final equity (4042.97 / 2790.85)Compare, Updating data would inevitably trigger an error; 
#    Now compare with 05 Daily equity just saved by the notebook risk_equity_history.csv Compare day by day on common dates
# ------------------------------------------------------------

ref_file = ROOT / "results" / "tables" / "risk_equity_history.csv"

if ref_file.exists():
    ref = pd.read_csv(ref_file, index_col="Date", parse_dates=True)
    for name, col in [("V0 Original", "Original"), ("V1 Inverse Vol", "Risk_Adjusted")]:
        common = equity_results[name].index.intersection(ref.index)
        assert len(common) > 0, f"{name}: and05Equity series have no common dates"
        assert np.allclose(
            equity_results[name].loc[common].values,
            ref[col].loc[common].values,
            rtol=1e-5
        ), f"{name} and05Results do not match: First rerun using the same data 05, Then run this file"
    print(f"\nV0, V1 and05Backtest consistency check passed! (Common dates {len(common)} days)")
else:
    print("\n⚠ Not found 05 notebooks risk_equity_history.csv, Skip V0/V1 Consistency check (Please run first 05)")


# ------------------------------------------------------------
# 6. Set backtest periods
# ------------------------------------------------------------

periods = {
    "IS": (ENGINE["IS_START"], ENGINE["IS_END"]),
    "OOS": (ENGINE["OOS_START"], ENGINE["OOS_END"]),
    "OOS ex-2020": ("2021-01-01", ENGINE["OOS_END"])
}


# ------------------------------------------------------------
# 7. Calculate performance metrics
# ------------------------------------------------------------

results = []

for name, equity in equity_results.items():

    for period, (start, end) in periods.items():

        performance = metrics(
            equity,
            a=start,
            b=end
        )

        results.append({
            "Strategy": name,
            "Period": period,
            **performance
        })


v2_performance = pd.DataFrame(results)


# ------------------------------------------------------------
# 8. Display three performance comparisons
# ------------------------------------------------------------

for period in periods:

    print(f"\n========== {period} ==========")

    display(
        v2_performance[
            v2_performance["Period"] == period
        ].round(4)
    )


In [ ]:
# ============================================================
# V2 - Step 6: Target volatility sensitivity tests
# ============================================================

TARGET_VOLS = [0.30, 0.40, 0.50, 0.60]

target_results = []
target_weights = {}
target_equities = {}

RISK_ASSETS = [etf for etf in ETFS if etf != "BSV"]

# ------------------------------------------------------------
# 1. Test different target volatilities
# ------------------------------------------------------------

for target_vol in TARGET_VOLS:

    print(f"Testing target volatility: {target_vol:.0%}")

    # Based onV1Estimated volatility determines scaling factors
    scale = (target_vol / portfolio_vol).clip(upper=1.0)

    # FromV1Weights begin changing
    weights = v1_weights.copy()

    # Scale non-BSVAssets
    weights[RISK_ASSETS] = (
        v1_weights[RISK_ASSETS]
        .mul(scale, axis=0)
    )

    # Allocate all remaining capital toBSV
    weights["BSV"] = (
        1.0 - weights[RISK_ASSETS].sum(axis=1)
    )

    # --------------------------------------------------------
    # 2. Check weights
    # --------------------------------------------------------

    assert weights.notna().all().all()

    assert (weights >= -1e-10).all().all()

    assert np.allclose(
        weights.sum(axis=1),
        1.0
    )

    # Save weights for each target
    target_weights[target_vol] = weights

    # --------------------------------------------------------
    # 3. Use the existing backtest function
    # --------------------------------------------------------

    equity, diag = backtest_weighted_portfolio(
        O,
        C,
        weights,
        start=IS_START
    )

    target_equities[target_vol] = equity

    # --------------------------------------------------------
    # 4. Record performance by period
    # --------------------------------------------------------

    for period, (start, end) in periods.items():

        performance = metrics(
            equity,
            a=start,
            b=end
        )

        target_results.append({
            "Target Vol": target_vol,
            "Period": period,
            "Average BSV": weights["BSV"].mean(),
            "Average TQQQ": weights["TQQQ"].mean(),
            **performance
        })

    print(f"Target volatility {target_vol:.0%} Tests complete!")


# ------------------------------------------------------------
# 5. Summarize results
# ------------------------------------------------------------

target_comparison = pd.DataFrame(target_results)

print("\n========== All test results ==========")

display(
    target_comparison.round(4)
)


# ------------------------------------------------------------
# 6. View separatelyIS / OOS / OOS ex-2020
# ------------------------------------------------------------

for period in periods:

    print(f"\n========== {period} ==========")

    display(
        target_comparison[
            target_comparison["Period"] == period
        ].round(4)
    )


# ------------------------------------------------------------
# 7. Check30%Reproduces the previousV2Results
# ------------------------------------------------------------

print("\n========== 30%Target consistency check ==========")

same_weights = np.allclose(
    target_weights[0.30][ETFS].values,
    v2_weights[ETFS].values
)

print("30%Target weights andStep 3Matches: ", same_weights)

assert same_weights, "30%Target failed to reproduce the previousV2Weights!"

assert np.isclose(
    target_equities[0.30].iloc[-1],
    equity_results["V2 BSV Targeting"].iloc[-1],
    rtol=1e-8
), "30%Target failed to reproduce the previousV2Equity!"

print("30%Target equity consistency check passed!")

In [ ]:
# Check the latest valid date of each dataset

print("PriceCSVLast row: ", prices.index.max())

print(
    "SixETFLatest complete price date: ",
    prices["Adj Close"][ETFS].dropna().index.max()
)

print("Latest original ensemble date: ", raw_weights.index.max())

print("V1Latest weight date: ", v1_weights.index.max())

print("V2Latest weight date: ", v2_weights.index.max())

print(
    "V0Latest equity date: ",
    equity_results["V0 Original"].index.max()
)

In [ ]:
# ============================================================
# V2 - Step 7: Export V2 (Target volatility TARGET_VOL)Daily weights
# For 07_final_strategy (Final strategy backtest)and 08_live_oct_09 (Live tracking ledger)Load
# ============================================================

SIGNAL_DIR = ROOT / "results" / "signals"
SIGNAL_DIR.mkdir(parents=True, exist_ok=True)

v2_out = v2_weights[ETFS].copy()      # Step 3 Generated V2 Weights (Target volatility = TARGET_VOL)
v2_out.index.name = "Date"

assert np.allclose(v2_out.sum(axis=1), 1.0)
assert (v2_out >= -1e-10).all().all()

v2_out.to_csv(SIGNAL_DIR / "v2_weights_history.csv", encoding="utf-8-sig")

print("Saved V2 Daily weights: ", SIGNAL_DIR / "v2_weights_history.csv")
print(f"Target volatility: {TARGET_VOL:.0%} | Period: {v2_out.index.min().date()} ~ {v2_out.index.max().date()}")
print("\nLatest daily weights (%): "); display((v2_out.iloc[-1] * 100).round(2).to_frame("Weight (%)"))
